In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

In [2]:
load_dotenv()           # reads OPENAI_API_KEY from .env

True

In [3]:
# create an OpenAI client using the API key from .env
client = OpenAI(base_url = "https://openrouter.ai/api/v1", api_key=os.getenv("OPENAI_API_KEY")) 

In [4]:
# model 1: stealth/ox-alpha
liquid_client = client.chat.completions.create(
    model="liquid/lfm-2.5-2.6b:free",
    messages=[
        {"role": "system", "content": "You are a witty travel guide."},
        {"role": "user", "content": "Suggest one thing to do in Bangalore."},
    ],
)

print("── GPT says:\n", liquid_client.choices[0].message.content)
print("── Model used:", liquid_client.model)

── GPT says:
 Ah, Bangalore — the City of Lakes, the Silicon Valley of South India, and home to some of the most dedicated chai-potions on earth! If you only have time for one thing, make it this:

## **Stroll through Cubbon Park & Visit Lalbagh Botanical Garden**

Here's why this is your golden ticket:

- **Cubbon Park** — Designed by the legendary British architect Henry M. Vignes in 1926, this sprawling green oasis is where the city breathes. Walk past the iconic fountain, find a quiet bench under a banyan tree, and let the gentle hum of traffic fade into the rustle of leaves. It's free, it's serene, and it's the perfect introduction to how Bangalore balances urban life with nature.

- **Lalbagh Botanical Garden** — Just a stone's throw from the park, this garden is a riot of color, fragrance, and architectural whimsy. Climb the giant waterfall (yes, really), admire the intricate marble statues, and lose yourself among the roses, jasmine, and exotic blooms. It's been declared one of

In [5]:
# model 2: Nvidia Model
nvidia_client = client.chat.completions.create(
    model="nvidia/nemotron-3.5-lightning:free",
    messages=[
        {"role": "system", "content": "You are a witty travel guide."},
        {"role": "user", "content": "Suggest one thing to do in Bangalore."},
    ],
)

print("── GPT says:\n", nvidia_client.choices[0].message.content)
print("── Model used:", nvidia_client.model)

── GPT says:
 One thing: Master the art of the Bangalore filter coffee. Hunt down a heritage joint—MTR, Vidyarthi Bhavan, or a so-tiny-it-doesn't-have-a-sign hole-in-the-wall—and order like you mean it. Sip the frothy,-cardamom-kissed brew from a steel tumbler and saucer while mentally competing with the locals on who has the more authentic "this isn't how it was in '95" story. It’s not just a caffeine hit; it’s a rite of passage, a love language spoken in aluminium, and the only thing capable of sustaining you through a day of nodding seriously at startup pitch decks. Bonus: you’ll finally understand why every Bangaloreans’ personality profile includes a mandatory "coffee before talking" disclaimer.
── Model used: nvidia/nemotron-3.5-lightning:free


In [6]:
# Give the model eyes — scrape a website LLMs only know what's in their training data.  To summarize
#  a *live* page we fetch it ourselves, strip out noise
#  (scripts, navbars, footers), and hand the clean text to GPT.
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

def fetch_website_contents(url):
    if not url.startswith(("http://", "https://")):
        url = "https://" + url

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        return f"Could not fetch the website. Error: {e}"

    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.title.string if soup.title else "No title"

    for tag in soup(["script", "style", "nav", "footer", "header", "img", "input"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)
    return f"Title: {title}\n\nPage contents:\n{text}"

In [7]:
# Summarize the scraped content with GPT
#  Now we chain the two pieces together: fetch() → summarize().
#  The system prompt keeps the model focused on content,
#  and we ask for markdown so the output renders nicely later.

SYSTEM_PROMPT = """You analyze the contents of a website and
give a short, friendly summary. Ignore navigation menus.
Respond in markdown."""

def summarize_website(url):
    website = fetch_website_contents(url)
    response =  client.chat.completions.create(
        model = "nvidia/nemotron-3-ultra-550b-a55b:free",
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Summarize this website:\n\n{website}"},
        ],)

    return response.choices[0].message.content